In [1]:
from pathlib import Path
import pandas as pd

# 数据所在文件夹
data_dir = Path(r"C:\Users\18208\Sephora_Project")

# 找到所有评论文件
review_files = sorted(data_dir.glob("reviews_*.csv"))

print("找到的评论文件：")
for file in review_files:
    print(file.name)

# 依次读取并纵向合并
review_list = []

for file in review_files:
    df_part = pd.read_csv(file, low_memory=False)
    review_list.append(df_part)
    print(file.name, df_part.shape)

reviews = pd.concat(
    review_list,
    ignore_index=True
)

print("评论总表形状：", reviews.shape)
print(reviews.head())

找到的评论文件：
reviews_0-250.csv
reviews_1250-end.csv
reviews_250-500.csv
reviews_500-750.csv
reviews_750-1250.csv
reviews_0-250.csv (602130, 19)
reviews_1250-end.csv (49977, 19)
reviews_250-500.csv (206725, 19)
reviews_500-750.csv (116262, 19)
reviews_750-1250.csv (119317, 19)
评论总表形状： (1094411, 19)
   Unnamed: 0    author_id  rating  is_recommended  helpfulness  \
0           0   1741593524       5             1.0          1.0   
1           1  31423088263       1             0.0          NaN   
2           2   5061282401       5             1.0          NaN   
3           3   6083038851       5             1.0          NaN   
4           4  47056667835       5             1.0          NaN   

   total_feedback_count  total_neg_feedback_count  total_pos_feedback_count  \
0                     2                         0                         2   
1                     0                         0                         0   
2                     0                         0                

In [3]:
products = pd.read_csv(
    data_dir / "product_info.csv",
    low_memory=False
)

print("商品表形状：", products.shape)
print(products.head())

商品表形状： (8494, 27)
  product_id               product_name  brand_id brand_name  loves_count  \
0    P473671    Fragrance Discovery Set      6342      19-69         6320   
1    P473668    La Habana Eau de Parfum      6342      19-69         3827   
2    P473662  Rainbow Bar Eau de Parfum      6342      19-69         3253   
3    P473660       Kasbah Eau de Parfum      6342      19-69         3018   
4    P473658  Purple Haze Eau de Parfum      6342      19-69         2691   

   rating  reviews            size                      variation_type  \
0  3.6364     11.0             NaN                                 NaN   
1  4.1538     13.0  3.4 oz/ 100 mL  Size + Concentration + Formulation   
2  4.2500     16.0  3.4 oz/ 100 mL  Size + Concentration + Formulation   
3  4.4762     21.0  3.4 oz/ 100 mL  Size + Concentration + Formulation   
4  3.2308     13.0  3.4 oz/ 100 mL  Size + Concentration + Formulation   

  variation_value  ... online_only out_of_stock  sephora_exclusive  \
0   

In [5]:
product_extra = products[[
    "product_id",
    "loves_count",
    "ingredients",
    "limited_edition",
    "new",
    "online_only",
    "out_of_stock",
    "sephora_exclusive",
    "highlights",
    "primary_category",
    "secondary_category",
    "tertiary_category"
]].drop_duplicates(subset=["product_id"])

df = reviews.merge(
    product_extra,
    on="product_id",
    how="left",
    validate="many_to_one"
)

print("合并后的数据形状：", df.shape)
print(df.head())

合并后的数据形状： (1094411, 30)
   Unnamed: 0    author_id  rating  is_recommended  helpfulness  \
0           0   1741593524       5             1.0          1.0   
1           1  31423088263       1             0.0          NaN   
2           2   5061282401       5             1.0          NaN   
3           3   6083038851       5             1.0          NaN   
4           4  47056667835       5             1.0          NaN   

   total_feedback_count  total_neg_feedback_count  total_pos_feedback_count  \
0                     2                         0                         2   
1                     0                         0                         0   
2                     0                         0                         0   
3                     0                         0                         0   
4                     0                         0                         0   

  submission_time                                        review_text  ...  \
0      2023-02-01  I 

In [9]:
df_clean = df.copy()

print("清洗前：", df_clean.shape)

# 1. 删除导出时产生的无意义序号列
df_clean = df_clean.drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

# 2. 日期格式转换
df_clean["submission_time"] = pd.to_datetime(
    df_clean["submission_time"],
    errors="coerce"
)

# 3. 评分转换为数值
df_clean["rating"] = pd.to_numeric(
    df_clean["rating"],
    errors="coerce"
)

# 只保留1-5分的正常评分
df_clean = df_clean[
    df_clean["rating"].between(1, 5)
].copy()

df_clean["rating"] = df_clean["rating"].astype("Int8")

# 4. 删除缺少核心字段的数据
df_clean = df_clean.dropna(
    subset=["product_id", "review_text"]
)

# 5. 删除空白评论
df_clean["review_text"] = (
    df_clean["review_text"]
    .astype("string")
    .str.strip()
)

df_clean = df_clean[
    df_clean["review_text"].ne("")
].copy()

# 6. 删除重复评论
duplicate_keys = [
    "author_id",
    "product_id",
    "submission_time",
    "review_text"
]

df_clean = df_clean.drop_duplicates(
    subset=duplicate_keys,
    keep="first"
)

# 7. 清理评论文本
df_clean["review_text_clean"] = (
    df_clean["review_text"]
    .str.lower()
    .str.replace(r"<[^>]+>", " ", regex=True)       # HTML标签
    .str.replace(r"http\S+|www\.\S+", " ", regex=True)  # 网址
    .str.replace(r"\s+", " ", regex=True)           # 多余空格
    .str.strip()
)

# 8. 评论标题缺失不删除，填为空字符串
df_clean["review_title"] = (
    df_clean["review_title"]
    .fillna("")
    .astype("string")
    .str.strip()
)

# 9. 推荐字段保留缺失值
df_clean["is_recommended"] = pd.to_numeric(
    df_clean["is_recommended"],
    errors="coerce"
).astype("Int8")

# 10. 用户属性缺失填Unknown
user_fields = [
    "skin_tone",
    "eye_color",
    "skin_type",
    "hair_color"
]

for col in user_fields:
    df_clean[col] = (
        df_clean[col]
        .fillna("Unknown")
        .astype("string")
        .str.strip()
        .replace("", "Unknown")
    )

# 11. 增加日期字段
df_clean["review_year"] = (
    df_clean["submission_time"].dt.year.astype("Int16")
)

df_clean["review_month"] = (
    df_clean["submission_time"].dt.month.astype("Int8")
)

# 12. 增加评论长度
df_clean["review_word_count"] = (
    df_clean["review_text_clean"]
    .str.split()
    .str.len()
    .astype("Int32")
)

# 13. 根据评分建立基础情感标签
# 注意：这是评分标签，不是后续NLP模型预测结果
df_clean["rating_sentiment"] = np.select(
    [
        df_clean["rating"].isin([1, 2]),
        df_clean["rating"].eq(3),
        df_clean["rating"].isin([4, 5])
    ],
    [
        "Negative",
        "Neutral",
        "Positive"
    ],
    default="Unknown"
)

# 14. 重新计算评论有用率
df_clean["helpfulness_calc"] = np.where(
    df_clean["total_feedback_count"] > 0,
    df_clean["total_pos_feedback_count"]
    / df_clean["total_feedback_count"],
    np.nan
)

print("清洗后：", df_clean.shape)

清洗前： (1094411, 30)
清洗后： (1092444, 35)


In [11]:
print("\n评分分布：")
print(
    df_clean["rating"]
    .value_counts()
    .sort_index()
)

print("\n情感标签分布：")
print(
    df_clean["rating_sentiment"]
    .value_counts()
)

print("\n肤质分布：")
print(
    df_clean["skin_type"]
    .value_counts()
    .head(10)
)

print("\n评论长度统计：")
print(
    df_clean["review_word_count"]
    .describe()
)

print("\n日期范围：")
print(
    df_clean["submission_time"].min(),
    "至",
    df_clean["submission_time"].max()
)

print("\n关键字段缺失值：")
print(
    df_clean[[
        "product_id",
        "product_name",
        "brand_name",
        "rating",
        "review_text_clean",
        "submission_time",
        "skin_type"
    ]].isna().sum()
)


评分分布：
rating
1     61081
2     52935
3     81729
4    199143
5    697556
Name: count, dtype: Int64

情感标签分布：
rating_sentiment
Positive    896699
Negative    114016
Neutral      81729
Name: count, dtype: int64

肤质分布：
skin_type
combination    543573
dry            185597
normal         131643
oily           120272
Unknown        111359
Name: count, dtype: Int64

评论长度统计：
count    1092444.0
mean     60.536865
std      43.479709
min            0.0
25%           32.0
50%           50.0
75%           76.0
max         1216.0
Name: review_word_count, dtype: Float64

日期范围：
2008-08-28 00:00:00 至 2023-03-21 00:00:00

关键字段缺失值：
product_id           0
product_name         0
brand_name           0
rating               0
review_text_clean    0
submission_time      0
skin_type            0
dtype: int64


In [17]:
df_clean = df_clean[
    df_clean["review_word_count"] > 0
].copy()

print("处理后的数据形状：", df_clean.shape)

处理后的数据形状： (1092435, 35)


In [19]:
df_nlp = df_clean[
    df_clean["review_word_count"] >= 5
].copy()

print("完整清洗数据：", df_clean.shape)
print("NLP分析数据：", df_nlp.shape)

完整清洗数据： (1092435, 35)
NLP分析数据： (1090758, 35)


In [21]:
df_clean.to_parquet(
    "sephora_reviews_clean.parquet",
    index=False
)

df_nlp.to_parquet(
    "sephora_reviews_nlp.parquet",
    index=False
)

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

data_dir = Path(r"C:\Users\18208\Sephora_Project")

df_clean = pd.read_parquet(
    data_dir / "sephora_reviews_clean.parquet"
)

df_nlp = pd.read_parquet(
    data_dir / "sephora_reviews_nlp.parquet"
)

print("完整数据：", df_clean.shape)
print("NLP数据：", df_nlp.shape)
print(df_nlp.head())

完整数据： (1092435, 35)
NLP数据： (1090758, 35)
     author_id  rating  is_recommended  helpfulness  total_feedback_count  \
0   1741593524       5               1          1.0                     2   
1  31423088263       1               0          NaN                     0   
2   5061282401       5               1          NaN                     0   
3   6083038851       5               1          NaN                     0   
4  47056667835       5               1          NaN                     0   

   total_neg_feedback_count  total_pos_feedback_count submission_time  \
0                         0                         2      2023-02-01   
1                         0                         0      2023-03-21   
2                         0                         0      2023-03-21   
3                         0                         0      2023-03-20   
4                         0                         0      2023-03-20   

                                         review_text  \
0

In [29]:
category_summary = (
    df_nlp
    .groupby("secondary_category")
    .agg(
        评论数量=("review_text_clean", "size"),
        商品数量=("product_id", "nunique"),
        品牌数量=("brand_name", "nunique"),
        平均评分=("rating", "mean"),
        负面评论数=(
            "rating_sentiment",
            lambda x: (x == "Negative").sum()
        )
    )
    .reset_index()
)

category_summary["负面率(%)"] = (
    category_summary["负面评论数"]
    / category_summary["评论数量"]
    * 100
).round(2)

category_summary["平均评分"] = (
    category_summary["平均评分"].round(2)
)

category_summary = category_summary.sort_values(
    "评论数量",
    ascending=False
)

category_summary.head(15)

,secondary_category,评论数量,商品数量,品牌数量,平均评分,负面评论数,负面率(%)
6,Moisturizers,296510,538,103,4.32,29120,9.82
10,Treatments,221344,452,92,4.3,22345,10.10
0,Cleansers,199809,354,93,4.34,19589,9.80
5,Mini Size,85241,110,44,4.29,9281,10.89
1,Eye Care,74813,184,83,4.18,8767,11.72
4,Masks,70311,165,60,4.34,7093,10.09
3,Lip Balms & Treatments,61257,61,38,4.33,6579,10.74
9,Sunscreen,41071,108,41,4.17,5672,13.81
11,Value & Gift Sets,12068,170,72,4.27,1365,11.31
7,Self Tanners,11911,53,9,4.08,1908,16.02


选 Sunscreen（防晒）。
原因很直接：
- 41,071条评论，样本量完全够
- 108款商品、41个品牌，可以进行品牌和产品比较
- 负面率13.81%，在主流护肤品类中最高
- 防晒痛点具体：泛白、搓泥、油腻、闷痘、熏眼、刺激、气味、防晒效果、价格
- 比“保湿是最大需求”更容易产出真正有价值的产品建议
项目正式定义为：
Sephora防晒产品用户痛点与市场机会分析：基于4万条用户评论，定位导致低评分的核心问题，并识别高潜产品定位。

最终要回答三个问题：
1. 防晒产品为什么比其他主流护肤品类更容易获得差评？
2. 泛白、油腻、搓泥、刺激、闷痘等问题中，哪个最影响评分和推荐？
3. 哪些品牌或产品解决得最好，新产品应该面向什么人群、具备什么卖点？

## 建立防晒专项数据集，看防晒下面是否还有细分类：

In [42]:
df_sun = df_nlp[
    df_nlp["secondary_category"]
    .eq("Sunscreen")
].copy()

print("防晒评论数：", len(df_sun))
print("防晒商品数：", df_sun["product_id"].nunique())
print("防晒品牌数：", df_sun["brand_name"].nunique())
print("平均评分：", round(df_sun["rating"].mean(), 2))
print(
    "负面率：",
    round(
        df_sun["rating_sentiment"]
        .eq("Negative")
        .mean() * 100,
        2
    ),
    "%"
)
print(
    df_sun["tertiary_category"]
    .fillna("无细分类")
    .value_counts(dropna=False)
)

防晒评论数： 41071
防晒商品数： 108
防晒品牌数： 41
平均评分： 4.17
负面率： 13.81 %
tertiary_category
Face Sunscreen    39026
Body Sunscreen     1296
无细分类                749
Name: count, dtype: int64


In [44]:
processed_dir = Path(
    r"C:\Users\18208\Sephora_Project"
)

# 读取未添加通用需求标签的数据
df_base = pd.read_parquet(
    processed_dir / "sephora_reviews_nlp.parquet"
)

# 只保留面部防晒
df_face = df_base[
    df_base["tertiary_category"]
    .eq("Face Sunscreen")
].copy()

print("面部防晒评论数：", len(df_face))
print("商品数：", df_face["product_id"].nunique())
print("品牌数：", df_face["brand_name"].nunique())
print("字段数：", df_face.shape[1])
df_face.to_parquet(
    processed_dir / "face_sunscreen_reviews.parquet",
    index=False
)

print("面部防晒数据保存完成")

面部防晒评论数： 39026
商品数： 95
品牌数： 41
字段数： 35
面部防晒数据保存完成


## 从面部防晒的正负面评论中，自动找出“更容易出现在差评里的词和词组”

In [50]:
from sklearn.feature_extraction.text import (
    CountVectorizer,
    ENGLISH_STOP_WORDS
)

# 1. 合并评论标题和正文
analysis_text = (
    df_face["review_title"]
    .fillna("")
    .astype(str)
    + " "
    + df_face["review_text_clean"]
    .fillna("")
    .astype(str)
)

# 2. 统一英文撇号，并处理否定缩写
analysis_text = (
    analysis_text
    .str.lower()
    .str.replace("’", "'", regex=False)
    .str.replace(r"n't\b", " not", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# 3. 英文停用词
# 保留no、not、never等否定词
custom_stopwords = (
    set(ENGLISH_STOP_WORDS)
    - {"no", "not", "nor", "never"}
)

# 删除对防晒痛点识别帮助不大的通用词
custom_stopwords.update([
    "product",
    "products",
    "sunscreen",
    "sunscreens",
    "skin",
    "face",
    "sephora",
    "received",
    "complimentary",
    "review",
    "sample",
    "use",
    "used",
    "using",
    "really",
    "just",
    "like"
])

# 4. 提取单词和二元词组
vectorizer = CountVectorizer(
    stop_words=sorted(custom_stopwords),
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.80,
    max_features=20000,
    binary=True,
    strip_accents="unicode"
)

X = vectorizer.fit_transform(
    analysis_text
)

terms = np.array(
    vectorizer.get_feature_names_out()
)

print("提取的词和词组数量：", len(terms))
print("文本矩阵形状：", X.shape)

提取的词和词组数量： 16168
文本矩阵形状： (39026, 16168)


In [52]:
# 5. 正负面评论位置
negative_mask = (
    df_face["rating_sentiment"]
    .eq("Negative")
    .to_numpy()
)

positive_mask = (
    df_face["rating_sentiment"]
    .eq("Positive")
    .to_numpy()
)

# 6. 每个词在正负面评论中的出现次数
negative_count = np.asarray(
    X[negative_mask].sum(axis=0)
).ravel()

positive_count = np.asarray(
    X[positive_mask].sum(axis=0)
).ravel()

# 7. 计算出现比例
negative_rate = (
    negative_count
    / negative_mask.sum()
    * 100
)

positive_rate = (
    positive_count
    / positive_mask.sum()
    * 100
)

# 8. 建立词组对比表
term_stats = pd.DataFrame({
    "词语或词组": terms,
    "负面评论出现数": negative_count.astype(int),
    "正面评论出现数": positive_count.astype(int),
    "负面出现率(%)": negative_rate,
    "正面出现率(%)": positive_rate
})

# 负面评论相对正面评论高出多少个百分点
term_stats["负面倾向差值"] = (
    term_stats["负面出现率(%)"]
    - term_stats["正面出现率(%)"]
)

# 负面评论中出现概率是正面评论的多少倍
term_stats["负面倾向倍数"] = (
    (term_stats["负面出现率(%)"] + 0.01)
    / (term_stats["正面出现率(%)"] + 0.01)
)

# 至少在20条负面评论中出现，避免极少数词干扰
negative_terms = (
    term_stats[
        term_stats["负面评论出现数"] >= 20
    ]
    .sort_values(
        "负面倾向差值",
        ascending=False
    )
    .head(40)
    .copy()
)

number_columns = [
    "负面出现率(%)",
    "正面出现率(%)",
    "负面倾向差值",
    "负面倾向倍数"
]

negative_terms[number_columns] = (
    negative_terms[number_columns]
    .round(2)
)

negative_terms.reset_index(
    drop=True
)

,词语或词组,负面评论出现数,正面评论出现数,负面出现率(%),正面出现率(%),负面倾向差值,负面倾向倍数
0,wanted,786,384,14.45,1.27,13.18,11.30
1,not,4167,20200,76.59,66.79,9.80,1.15
2,bad,578,672,10.62,2.22,8.40,4.76
3,work,656,1136,12.06,3.76,8.30,3.20
4,wanted love,451,26,8.29,0.09,8.20,86.48
5,pills,469,134,8.62,0.44,8.18,19.05
6,did,1169,4174,21.49,13.80,7.68,1.56
7,did not,1002,3435,18.42,11.36,7.06,1.62
8,disappointed,392,148,7.20,0.49,6.72,14.45
9,not recommend,384,247,7.06,0.82,6.24,8.55


In [54]:
export_table = (
    negative_terms
    .reset_index(drop=True)
    .copy()
)

export_table.insert(
    0,
    "排名",
    range(1, len(export_table) + 1)
)

export_path = (
    processed_dir
    / "face_sunscreen_negative_terms.xlsx"
)

export_table.to_excel(
    export_path,
    index=False
)

print("导出完成：", export_path)

导出完成： C:\Users\18208\Sephora_Project\face_sunscreen_negative_terms.xlsx


## 差评文本 → 自动聚类主题 → 查看每个主题关键词和代表评论 → 人工命名为泛白、搓泥、刺激等痛点。

In [58]:
##第一步：提取负面评论

negative_face = df_face[
    df_face["rating_sentiment"]
    .eq("Negative")
].copy()

print("面部防晒负面评论数：", len(negative_face))

面部防晒负面评论数： 5441


In [60]:
##第二步：训练NMF主题模型

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)

from sklearn.decomposition import NMF

# 情绪词只能说明“不满意”，无法说明具体不满意什么
emotion_words = {
    "wanted", "love", "bad", "worst",
    "horrible", "terrible", "awful",
    "disappointed", "unfortunately",
    "recommend", "reviews", "review",
    "thought", "maybe", "better",
    "product", "products", "sunscreen",
    "sunscreens", "sephora", "sample",
    "received", "complimentary",
    "use", "used", "using",
    "really", "just", "like",
    "did", "got", "time"
}

topic_stopwords = (
    set(ENGLISH_STOP_WORDS)
    - {"no", "not", "nor", "never"}
)

topic_stopwords.update(emotion_words)

# 合并标题和正文
negative_text = (
    negative_face["review_title"]
    .fillna("")
    .astype(str)
    + " "
    + negative_face["review_text_clean"]
    .fillna("")
    .astype(str)
)

negative_text = (
    negative_text
    .str.lower()
    .str.replace("’", "'", regex=False)
    .str.replace(r"n't\b", " not", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# 文本转为TF-IDF特征
tfidf_vectorizer = TfidfVectorizer(
    stop_words=sorted(topic_stopwords),
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.75,
    max_features=6000,
    sublinear_tf=True,
    strip_accents="unicode"
)

X_tfidf = tfidf_vectorizer.fit_transform(
    negative_text
)

# 暂时设定为8个主题
nmf_model = NMF(
    n_components=8,
    init="nndsvda",
    random_state=42,
    max_iter=500
)

topic_weights = nmf_model.fit_transform(
    X_tfidf
)

print("TF-IDF矩阵：", X_tfidf.shape)
print("主题权重矩阵：", topic_weights.shape)

TF-IDF矩阵： (5441, 2564)
主题权重矩阵： (5441, 8)


In [62]:
##第三步：查看每个主题的关键词

feature_names = np.array(
    tfidf_vectorizer.get_feature_names_out()
)

topic_keyword_rows = []

for topic_id, topic_vector in enumerate(
    nmf_model.components_
):

    top_indices = topic_vector.argsort()[-15:][::-1]
    top_keywords = feature_names[top_indices]

    topic_keyword_rows.append({
        "主题编号": topic_id,
        "主题关键词": "、".join(top_keywords)
    })

topic_keywords = pd.DataFrame(
    topic_keyword_rows
)

topic_keywords

,主题编号,主题关键词
0,0,face、greasy、look、day、spf、felt、makeup、no、good、sticky、sun、wear、way、feel、texture
1,1,white、cast、white cast、leaves、leaves white、left、left white、skin、cast not、cast face、leave、cast skin、leave white、no white、rub
2,2,oily、skin、oily skin、dry、dry skin、greasy、not oily、skin not、combination、look、super、skin oily、combo、combination skin、oil
3,3,pills、pilling、tried、crazy、glow、pills crazy、ve、no、glow recipe、recipe、ve tried、matter、no matter、moisturizer、pilled
4,4,smell、smells、spray、ca not、ca、strong、alcohol、setting、eyes、setting spray、smell not、not smell、scent、smelled、makeup
5,5,does、does not、work、not work、absorb、not absorb、makeup、skin does、blend、not blend、leave、make、skin、feel、not leave
6,6,worth、not worth、money、packaging、waste、price、waste money、bottle、buy、not buy、worth money、worth price、not waste、save、save money
7,7,sensitive、acne、sensitive skin、skin、prone、break、broke、acne prone、prone skin、not sensitive、breakouts、caused、breakout、bumps、skin not


In [64]:
##第四步：查看每个主题的代表评论

negative_face["topic_id"] = (
    topic_weights.argmax(axis=1)
)

negative_face["topic_strength"] = (
    topic_weights.max(axis=1)
)

for topic_id in range(8):

    print("\n")
    print("=" * 80)
    print("主题", topic_id)
    print(
        topic_keywords.loc[
            topic_keywords["主题编号"] == topic_id,
            "主题关键词"
        ].iloc[0]
    )

    example_reviews = (
        negative_face[
            negative_face["topic_id"] == topic_id
        ]
        .nlargest(
            3,
            "topic_strength"
        )["review_text_clean"]
    )

    for number, review in enumerate(
        example_reviews,
        start=1
    ):
        print(f"\n代表评论{number}：")
        print(review)
        



主题 0
face、greasy、look、day、spf、felt、makeup、no、good、sticky、sun、wear、way、feel、texture

代表评论1：
this stuff feels thick and sticky, and it left my face looking greasy. within minutes of application, my face was sweating when no other part of my body was! i could feel it on my face like a mask. not to mention the tint is this weird yellow-brown that looks awful on my fair skin. i’m so glad i tried this in the sun safety kit before i bought the full size.

代表评论2：
this felt ok on my face, but i didn’t really like the tint, as it changed the color of my face. it also looked very greasy on my skin.

代表评论3：
i love the glowscreen by this brand and happily wore makeup over it with or without foundation year long, and was hoping for a similar thin and “not sunscreen“ like feeling when putting this on as it is literally called the unseen sunscreen. especially because this is so heavily promoted online. i was disappointed this felt very silicone-like and the best comparison i have is that it felt lik

NMF主题不能直接作为最终标签，因为：
- 主题0和2都指向油腻、厚重，可以合并；
- 主题5混合了防晒无效、难吸收和妆前不兼容，需要拆开；
- 主题6混合了价格和包装，需要拆开；
- 主题7可以拆成爆痘与刺激两类。
最终形成10个有数据依据的面部防晒痛点：
标签	                   来源
油腻厚重	                主题0、2
泛白与肤色不匹配	        主题1
搓泥	                 主题3
气味问题	                  主题4
吸收及妆前兼容	          主题5
防晒效果不足	            主题5代表评论
价格与性价比	           主题6
包装问题           	    主题6
爆痘堵塞	                主题7
刺激与熏眼	          主题7及部分主题4

In [66]:
pain_keywords = {
    "greasy_heavy": [
        "greasy", "oily", "sticky", "tacky",
        "heavy", "thick", "occlusive",
        "silicone-like", "vaseline"
    ],

    "white_color_cast": [
        "white cast", "leaves white", "left white",
        "ashy", "chalky", "ghostly",
        "orange tint", "yellow tint",
        "wrong shade"
    ],

    "pilling": [
        "pill", "pills", "pilling", "pilled",
        "balls up", "balling up",
        "flakes off"
    ],

    "smell": [
        "smell", "smells", "smelled",
        "scent", "fragrance", "odor",
        "alcohol smell", "chemical smell"
    ],

    "absorb_makeup": [
        "not absorb", "does not absorb",
        "doesn't absorb", "won't absorb",
        "not blend", "does not blend",
        "doesn't blend", "hard to blend",
        "under makeup", "under my makeup",
        "foundation", "primer"
    ],

    "protection_failure": [
        "sunburn", "sunburned", "sunburnt",
        "burned in the sun", "burnt in the sun",
        "did not protect", "didn't protect",
        "does not protect", "doesn't protect",
        "no protection", "not protective"
    ],

    "price_value": [
        "not worth", "worth the money",
        "waste of money", "waste money",
        "overpriced", "too expensive",
        "price", "expensive", "money"
    ],

    "packaging": [
        "packaging", "bottle", "pump",
        "nozzle", "sprayer", "dispenser",
        "cap", "tube", "leak", "leaking"
    ],

    "breakout_clogging": [
        "breakout", "breakouts",
        "broke me out", "break me out",
        "acne", "pimple", "pimples",
        "clogged pore", "clogged pores",
        "tiny bumps", "milia"
    ],

    "irritation_eyes": [
        "irritated", "irritation",
        "rash", "allergic", "reaction",
        "redness", "stinging", "stings",
        "burning", "burns my eyes",
        "stings my eyes", "eye irritation",
        "watery eyes", "eyes water"
    ]
}

In [68]:
import re

negative_text_clean = (
    negative_face["review_text_clean"]
    .fillna("")
    .str.lower()
    .str.replace("’", "'", regex=False)
)

pain_columns = []

for pain, keywords in pain_keywords.items():

    keywords_sorted = sorted(
        keywords,
        key=len,
        reverse=True
    )

    keyword_pattern = "|".join(
        re.escape(keyword)
        for keyword in keywords_sorted
    )

    pattern = (
        rf"(?<!\w)(?:{keyword_pattern})(?!\w)"
    )

    column_name = "pain_" + pain
    pain_columns.append(column_name)

    negative_face[column_name] = (
        negative_text_clean
        .str.contains(
            pattern,
            regex=True,
            na=False
        )
        .astype("int8")
    )

In [70]:
pain_summary = pd.DataFrame({
    "负面评论数量":
        negative_face[pain_columns].sum(),

    "占全部负面评论比例(%)":
        negative_face[pain_columns].mean() * 100
})

pain_summary.index = (
    pain_summary.index
    .str.replace("pain_", "", regex=False)
)

pain_summary["负面评论数量"] = (
    pain_summary["负面评论数量"]
    .astype(int)
)

pain_summary["占全部负面评论比例(%)"] = (
    pain_summary["占全部负面评论比例(%)"]
    .round(2)
)

pain_summary = pain_summary.sort_values(
    "负面评论数量",
    ascending=False
)

pain_summary

,负面评论数量,占全部负面评论比例(%)
greasy_heavy,2120,38.96
smell,872,16.03
white_color_cast,860,15.81
pilling,803,14.76
absorb_makeup,738,13.56
price_value,735,13.51
packaging,643,11.82
breakout_clogging,609,11.19
irritation_eyes,349,6.41
protection_failure,125,2.30


In [72]:
raw_dir = Path(
    r"C:\Users\18208\Sephora_Project"
)

products = pd.read_csv(
    raw_dir / "product_info.csv",
    low_memory=False
)

product_meta = (
    products[[
        "product_id",
        "size",
        "variation_type",
        "variation_value",
        "rating",
        "reviews",
        "loves_count",
        "price_usd",
        "value_price_usd",
        "sale_price_usd"
    ]]
    .drop_duplicates("product_id")
    .rename(columns={
        "rating": "平台商品评分",
        "reviews": "平台评论总数",
        "loves_count": "收藏关注数",
        "price_usd": "商品价格USD",
        "value_price_usd": "套装价值USD",
        "sale_price_usd": "促销价格USD",
        "size": "产品规格",
        "variation_type": "规格类型",
        "variation_value": "规格值"
    })
)

print(product_meta.shape)
product_meta.head()

(8494, 10)


,product_id,产品规格,规格类型,规格值,平台商品评分,平台评论总数,收藏关注数,商品价格USD,套装价值USD,促销价格USD
0,P473671,NaN,NaN,NaN,3.6364,11.0,6320,35.0,NaN,NaN
1,P473668,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,4.1538,13.0,3827,195.0,NaN,NaN
2,P473662,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,4.2500,16.0,3253,195.0,NaN,NaN
3,P473660,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,4.4762,21.0,3018,195.0,NaN,NaN
4,P473658,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,3.2308,13.0,2691,195.0,NaN,NaN


In [74]:
product_analysis = (
    df_face
    .groupby(
        [
            "product_id",
            "product_name",
            "brand_name"
        ],
        dropna=False
    )
    .agg(
        样本评论数=("rating", "size"),
        评论用户数=("author_id", "nunique"),
        样本平均评分=("rating", "mean"),
        负面评论数=(
            "rating_sentiment",
            lambda x: x.eq("Negative").sum()
        ),
        推荐率=(
            "is_recommended",
            "mean"
        )
    )
    .reset_index()
)

product_analysis["负面率"] = (
    product_analysis["负面评论数"]
    / product_analysis["样本评论数"]
)

product_analysis["样本平均评分"] = (
    product_analysis["样本平均评分"].round(2)
)

product_analysis = product_analysis.merge(
    product_meta,
    on="product_id",
    how="left",
    validate="one_to_one"
)

In [76]:
pain_name_cn = {
    "pain_greasy_heavy": "油腻厚重",
    "pain_white_color_cast": "泛白及肤色不适配",
    "pain_pilling": "搓泥",
    "pain_smell": "气味",
    "pain_absorb_makeup": "吸收及妆前兼容",
    "pain_protection_failure": "防晒效果不足",
    "pain_price_value": "价格与性价比",
    "pain_packaging": "包装",
    "pain_breakout_clogging": "爆痘堵塞",
    "pain_irritation_eyes": "刺激及熏眼"
}

product_pain = (
    negative_face
    .groupby("product_id")[pain_columns]
    .sum()
)

product_pain["已识别痛点总次数"] = (
    product_pain[pain_columns].sum(axis=1)
)

product_pain["主要痛点字段"] = (
    product_pain[pain_columns]
    .idxmax(axis=1)
)

product_pain["主要痛点"] = (
    product_pain["主要痛点字段"]
    .map(pain_name_cn)
)

product_pain["主要痛点评论数"] = (
    product_pain[pain_columns]
    .max(axis=1)
)

# 如果一条痛点都没有识别出来
product_pain.loc[
    product_pain["已识别痛点总次数"].eq(0),
    "主要痛点"
] = "未识别"

product_pain = (
    product_pain
    .reset_index()
    .drop(columns=["主要痛点字段"])
)

product_analysis = product_analysis.merge(
    product_pain,
    on="product_id",
    how="left"
)

In [78]:
def get_top_segment(data, segment_column, output_name):

    segment_data = data[
        [
            "product_id",
            segment_column
        ]
    ].copy()

    segment_data[segment_column] = (
        segment_data[segment_column]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

    segment_data = segment_data[
        ~segment_data[segment_column].isin(
            ["Unknown", "", "nan", "None"]
        )
    ]

    segment_count = (
        segment_data
        .groupby(
            ["product_id", segment_column]
        )
        .size()
        .reset_index(name="人数")
    )

    segment_count["已知人群总数"] = (
        segment_count
        .groupby("product_id")["人数"]
        .transform("sum")
    )

    segment_count["人群占比"] = (
        segment_count["人数"]
        / segment_count["已知人群总数"]
    )

    top_segment = (
        segment_count
        .sort_values(
            ["product_id", "人数"],
            ascending=[True, False]
        )
        .drop_duplicates("product_id")
        [[
            "product_id",
            segment_column,
            "人群占比"
        ]]
        .rename(columns={
            segment_column: output_name,
            "人群占比": output_name + "占比"
        })
    )

    return top_segment

In [80]:
top_skin_type = get_top_segment(
    df_face,
    "skin_type",
    "主要肤质"
)

top_skin_tone = get_top_segment(
    df_face,
    "skin_tone",
    "主要肤色"
)

product_analysis = (
    product_analysis
    .merge(
        top_skin_type,
        on="product_id",
        how="left"
    )
    .merge(
        top_skin_tone,
        on="product_id",
        how="left"
    )
)

In [82]:
product_analysis = product_analysis[[
    "product_id",
    "product_name",
    "brand_name",
    "产品规格",
    "商品价格USD",
    "促销价格USD",
    "收藏关注数",
    "平台评论总数",
    "平台商品评分",
    "样本评论数",
    "评论用户数",
    "样本平均评分",
    "推荐率",
    "负面评论数",
    "负面率",
    "主要痛点",
    "主要痛点评论数",
    "主要肤质",
    "主要肤质占比",
    "主要肤色",
    "主要肤色占比"
]].sort_values(
    "样本评论数",
    ascending=False
)

product_analysis.head(20)

,product_id,product_name,brand_name,产品规格,商品价格USD,促销价格USD,收藏关注数,平台评论总数,平台商品评分,样本评论数,...,样本平均评分,推荐率,负面评论数,负面率,主要痛点,主要痛点评论数,主要肤质,主要肤质占比,主要肤色,主要肤色占比
25,P454380,Unseen Sunscreen SPF 40 PA+++,Supergoop!,2.5 oz/ 73.9 mL,48.0,NaN,302951,3262.0,4.2600,3258,...,4.26,0.81062,453,0.139042,油腻厚重,147.0,combination,0.540709,light,0.268666
33,P456398,Ultimate Sun Protector Lotion SPF 50+ Sunscreen,Shiseido,5.0 oz/ 150 mL,50.0,NaN,42257,2936.0,4.4864,2937,...,4.49,0.921008,115,0.039156,油腻厚重,83.0,combination,0.563848,light,0.269667
31,P456218,Glowscreen Sunscreen SPF 40 PA+++ with Hyaluronic Acid + Niacinamide,Supergoop!,1.7 oz Golden Hour / 50 mL,38.0,NaN,231004,2049.0,4.1513,2064,...,4.15,0.780039,316,0.153101,油腻厚重,82.0,combination,0.498263,light,0.287018
14,P429242,Clear Sunscreen Stick SPF 50+,Shiseido,0.7 oz/ 20 g,30.0,NaN,74272,1529.0,4.6566,1528,...,4.66,0.951571,28,0.018325,油腻厚重,13.0,combination,0.568421,light,0.265387
32,P456392,Daily UV Defense Invisible Broad Spectrum SPF 36 Sunscreen,innisfree,1.69 oz/ 50 mL,16.0,NaN,145749,1452.0,4.5393,1464,...,4.54,0.897541,101,0.068989,爆痘堵塞,34.0,combination,0.528145,light,0.253912
63,P482320,Mini Glowscreen Sunscreen SPF 40 PA+++ with Hyaluronic Acid + Niacinamide,Supergoop!,.68 oz Sunrise / 20 mL,22.0,NaN,64624,1439.0,3.9659,1455,...,3.97,0.723024,287,0.197251,油腻厚重,75.0,combination,0.469474,light,0.298195
11,P419222,Umbra Sheer Physical Daily Defense SPF 30,Drunk Elephant,3 oz/ 90 mL,34.0,NaN,46738,1362.0,3.3076,1361,...,3.31,0.559361,453,0.332843,油腻厚重,224.0,combination,0.536474,light,0.303846
69,P482739,Urban Environment Oil-Free Sunscreen Broad-Spectrum SPF 42,Shiseido,1.6 oz/ 50 mL,50.0,NaN,12870,1337.0,4.4533,1343,...,4.45,0.908414,58,0.043187,泛白及肤色不适配,29.0,combination,0.597160,light,0.304740
57,P481989,Watermelon Glow Niacinamide Sunscreen SPF 50,Glow Recipe,1.7 oz/ 50 mL,35.0,NaN,64453,1293.0,3.7610,1295,...,3.76,0.654054,329,0.254054,搓泥,218.0,combination,0.531915,light,0.287867
72,P482742,Urban Environment Fresh-Moisture Sunscreen Broad-Spectrum SPF 42,Shiseido,1.8 oz/ 50 mL,38.0,NaN,8629,1190.0,4.6311,1190,...,4.63,0.955462,16,0.013445,油腻厚重,9.0,combination,0.606751,light,0.309383


In [84]:
brand_analysis = (
    df_face
    .groupby("brand_name")
    .agg(
        商品数量=("product_id", "nunique"),
        样本评论数=("rating", "size"),
        评论用户数=("author_id", "nunique"),
        平均评分=("rating", "mean"),
        负面评论数=(
            "rating_sentiment",
            lambda x: x.eq("Negative").sum()
        ),
        推荐率=("is_recommended", "mean")
    )
    .reset_index()
)

brand_analysis["负面率"] = (
    brand_analysis["负面评论数"]
    / brand_analysis["样本评论数"]
)

brand_analysis["平均评分"] = (
    brand_analysis["平均评分"].round(2)
)

brand_heat = (
    product_analysis
    .groupby("brand_name")
    .agg(
        商品价格中位数USD=(
            "商品价格USD",
            "median"
        ),
        收藏关注总数=(
            "收藏关注数",
            "sum"
        ),
        平台评论总数=(
            "平台评论总数",
            "sum"
        )
    )
    .reset_index()
)

brand_analysis = (
    brand_analysis
    .merge(
        brand_heat,
        on="brand_name",
        how="left"
    )
    .sort_values(
        "样本评论数",
        ascending=False
    )
)

In [86]:
def build_customer_profile(
    review_data,
    negative_data,
    segment_column
):

    profile = (
        review_data
        .groupby(segment_column)
        .agg(
            评论数=("rating", "size"),
            用户数=("author_id", "nunique"),
            平均评分=("rating", "mean"),
            负面评论数=(
                "rating_sentiment",
                lambda x: x.eq("Negative").sum()
            ),
            推荐率=("is_recommended", "mean")
        )
        .reset_index()
    )

    profile["负面率"] = (
        profile["负面评论数"]
        / profile["评论数"]
    )

    segment_pain = (
        negative_data
        .groupby(segment_column)[pain_columns]
        .sum()
        .reset_index()
    )

    profile = profile.merge(
        segment_pain,
        on=segment_column,
        how="left"
    )

    profile["平均评分"] = (
        profile["平均评分"].round(2)
    )

    return profile

skin_type_profile = build_customer_profile(
    df_face,
    negative_face,
    "skin_type"
)

skin_tone_profile = build_customer_profile(
    df_face,
    negative_face,
    "skin_tone"
)

export_path = (
    processed_dir
    / "sephora_face_sunscreen_analysis.xlsx"
)

with pd.ExcelWriter(
    export_path,
    engine="openpyxl"
) as writer:

    pain_summary.to_excel(
        writer,
        sheet_name="痛点总览"
    )

    product_analysis.to_excel(
        writer,
        sheet_name="产品分析",
        index=False,
        freeze_panes=(1, 0)
    )

    brand_analysis.to_excel(
        writer,
        sheet_name="品牌分析",
        index=False,
        freeze_panes=(1, 0)
    )

    skin_type_profile.to_excel(
        writer,
        sheet_name="肤质人群",
        index=False,
        freeze_panes=(1, 0)
    )

    skin_tone_profile.to_excel(
        writer,
        sheet_name="肤色人群",
        index=False,
        freeze_panes=(1, 0)
    )

print("导出完成：", export_path)

导出完成： C:\Users\18208\Sephora_Project\sephora_face_sunscreen_analysis.xlsx


用用户评论找出面部防晒中表现最差、表现最好以及最值得改善的产品，并形成新品定位建议。

接下来只做三件事：
1. 找出高风险产品：评论量足够，但负面率明显高。
2. 找出标杆产品：评论量足够，评分高且负面率低。
3. 对比它们的价格、规格、主要人群和主要痛点，得出产品机会。

In [89]:
valid_products = product_analysis[
    product_analysis["样本评论数"] >= 100
].copy()

print("满足100条评论的产品数：", len(valid_products))

满足100条评论的产品数： 64


In [91]:
risk_products = (
    valid_products
    .sort_values(
        ["负面率", "样本评论数"],
        ascending=[False, False]
    )
    .head(10)
    [[
        "product_name",
        "brand_name",
        "产品规格",
        "商品价格USD",
        "样本评论数",
        "样本平均评分",
        "推荐率",
        "负面率",
        "主要痛点",
        "主要肤质",
        "主要肤色"
    ]]
)

risk_products

,product_name,brand_name,产品规格,商品价格USD,样本评论数,样本平均评分,推荐率,负面率,主要痛点,主要肤质,主要肤色
5,Ultra Sun Protection Lotion Broad Spectrum SPF 50+ WetForce for Sensitive Skin & Children,Shiseido,3.3 oz/ 100 mL,42.0,227,3.18,0.442953,0.396476,泛白及肤色不适配,combination,fair
10,Umbra Tinte Physical Daily Defense SPF 30,Drunk Elephant,2 oz/ 60 mL,36.0,842,3.3,0.519949,0.390736,油腻厚重,combination,light
19,Clean Screen Mineral SPF 30 Mattifying Face Sunscreen,REN Clean Skincare,NaN,40.0,414,3.35,0.52657,0.350242,油腻厚重,combination,light
4,Makeup Setting Spray Organic Sunscreen SPF 30,COOLA,1.5 oz,36.0,622,3.31,0.571429,0.347267,气味,combination,lightMedium
11,Umbra Sheer Physical Daily Defense SPF 30,Drunk Elephant,3 oz/ 90 mL,34.0,1361,3.31,0.559361,0.332843,油腻厚重,combination,light
54,The Silk Sunscreen Mineral Broad Spectrum SPF 50 PA++++ with Hyaluronic Acid and Niacinamide,Tatcha,1.7 oz/ 50 mL,62.0,484,3.4,0.56405,0.301653,包装,combination,light
16,Super Fluid Daily UV Defense Sunscreen Broad Spectrum SPF 50+,Kiehl's Since 1851,4.2 oz / 125 ml,68.0,125,3.62,0.576,0.288,气味,combination,lightMedium
17,Glow Stick Sunscreen SPF 50 PA++++,Supergoop!,0.7 oz/ 20 g,26.0,495,3.65,0.672727,0.264646,油腻厚重,combination,light
57,Watermelon Glow Niacinamide Sunscreen SPF 50,Glow Recipe,1.7 oz/ 50 mL,35.0,1295,3.76,0.654054,0.254054,搓泥,combination,light
66,(Re) Setting Refreshing Mist SPF 40,Supergoop!,3.4 oz/ 100 mL,34.0,382,3.64,0.696335,0.240838,油腻厚重,combination,lightMedium


In [93]:
benchmark_products = (
    valid_products
    .sort_values(
        [
            "负面率",
            "样本平均评分",
            "样本评论数"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .head(10)
    [[
        "product_name",
        "brand_name",
        "产品规格",
        "商品价格USD",
        "样本评论数",
        "样本平均评分",
        "推荐率",
        "负面率",
        "主要痛点",
        "主要肤质",
        "主要肤色"
    ]]
)

benchmark_products

,product_name,brand_name,产品规格,商品价格USD,样本评论数,样本平均评分,推荐率,负面率,主要痛点,主要肤质,主要肤色
94,Polyglutamic Acid Dewy Sunscreen SPF 30,The INKEY List,1.7 oz / 50 ml,14.99,100,4.69,0.97,0.0,NaN,combination,fair
89,Dynamic Skin Recovery SPF50 Moisturizer,Dermalogica,1.7 oz / 50 mL,79.00,266,4.81,0.977444,0.003759,未识别,combination,light
78,The True Cream - Aqua Bomb Sunscreen Broad Spectrum SPF 50,belif,NaN,38.00,423,4.52,0.945626,0.009456,爆痘堵塞,combination,light
88,Luxury Sun Ritual Pore Smoothing Sunscreen SPF 30,Saint Jane Beauty,1.7 oz / 50 mL,38.00,293,4.83,0.972696,0.010239,泛白及肤色不适配,combination,light
93,Max Vitamin D-Fense Sunscreen Serum Broad Spectrum SPF 50,Peter Thomas Roth,1.7 oz / 50 mL,42.00,172,4.81,0.982558,0.011628,泛白及肤色不适配,combination,light
72,Urban Environment Fresh-Moisture Sunscreen Broad-Spectrum SPF 42,Shiseido,1.8 oz/ 50 mL,38.00,1190,4.63,0.955462,0.013445,油腻厚重,combination,light
14,Clear Sunscreen Stick SPF 50+,Shiseido,0.7 oz/ 20 g,30.00,1528,4.66,0.951571,0.018325,油腻厚重,combination,light
68,Correct & Protect Broad Spectrum SPF 45| PA++++,Murad,1 oz/ 30 mL,69.00,467,4.74,0.963597,0.019272,油腻厚重,combination,light
62,Get Even Sun Milk SPF 33,Josie Maran,2 oz/ 60 mL,46.00,300,4.7,0.95,0.026667,油腻厚重,combination,light
43,Full Spectrum 360º Refreshing Water Cream Organic Face Sunscreen SPF 50,COOLA,1.5 oz,48.00,267,4.58,0.93633,0.029963,油腻厚重,combination,light


In [97]:
with pd.ExcelWriter(
    export_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    risk_products.to_excel(
        writer,
        sheet_name="高风险产品",
        index=False
    )

    benchmark_products.to_excel(
        writer,
        sheet_name="标杆产品",
        index=False
    )

print("高风险产品和标杆产品已写入Excel")

高风险产品和标杆产品已写入Excel


核心结论
面部防晒最大的差评来源不是防晒能力，而是日常使用体验：
- 油腻厚重：38.96%，最主要痛点
- 气味：16.03%
- 泛白及肤色不适配：15.81%
- 搓泥：14.76%
- 吸收及妆前兼容：13.56%
- 价格与性价比：13.51%
“防晒效果不足”只识别到2.30%。但不能据此认为防晒效果不重要，因为普通用户难以准确判断防护力，而且关键词规则可能漏识别。它只能说明：用户在评论中更常抱怨肤感和妆效。
哪些产品最值得关注
高风险产品的问题非常具体：
- Shiseido儿童及敏感肌防晒：负面率39.65%，主要问题是泛白。
- Drunk Elephant Umbra Tinte：负面率39.07%，主要问题是油腻厚重。
- Drunk Elephant Umbra Sheer：负面率33.28%，同样集中在油腻厚重。
- COOLA定妆防晒喷雾：负面率34.73%，主要问题是气味。
- Tatcha The Silk Sunscreen：负面率30.17%，主要问题是包装。
- Glow Recipe西瓜防晒：负面率25.41%，主要问题是搓泥。
这说明问题不是简单的“品牌不好”，而是具体配方、剂型和包装设计的问题。
例如Shiseido整体品牌平均评分4.47、负面率仅4.49%，属于表现优秀的品牌；但它的敏感肌儿童防晒单品负面率接近40%。所以必须分析到产品层级，不能只做品牌排名。
标杆产品
更可靠的标杆是评论量较大且负面率低的产品：
- Shiseido Urban Environment SPF42：1,190条评论，负面率1.34%。
- Shiseido Clear Sunscreen Stick：1,528条评论，负面率1.83%。
- Murad Correct & Protect：467条评论，负面率1.93%。
- belif Aqua Bomb Sunscreen：423条评论，负面率0.95%。
The INKEY List显示100条评论、0条负面评论，但样本量刚刚达到门槛，结论需要谨慎，不能直接说它是绝对最佳。
价格不是决定性因素
高风险产品价格中位数约36美元，标杆产品约40美元，全部有效产品中位数约38美元。
因此不能得出“越贵越差”或“便宜产品更好”。真正拉开评分差距的是：
- 油腻程度；
- 是否泛白；
- 是否搓泥；
- 气味；
- 包装；
- 妆前兼容性。
客户群体洞察
油性肌肤的痛点非常鲜明：
- 油皮差评中52.65%提到油腻厚重；
- 混合肌为39.82%；
- 干皮和中性肌约33.6%。
所以“轻薄、不油腻”尤其是油皮用户的核心需求。
肤色方面：
- 深肤色差评中39.50%提到泛白；
- tan肤色为30.05%；
- light肤色只有12.46%；
- fair肤色为13.94%。
这证明“不泛白、适配中深肤色”是一个真实且显著的产品机会。
最终产品机会
基于现有结果，可以提出一款：
面向油性、混合性及中深肤色用户的轻薄面部防晒：透明不泛白、不油腻、不搓泥、无明显香味，能够与底妆兼容，采用稳定易用的包装，规格约50mL，价格带约30—40美元。

不过Excel里的“主要肤质”和“主要肤色”不能直接当作产品目标人群，因为大多数产品都会显示 combination 和 light，只是这两类用户在原始数据中人数最多。最终报告应使用“某人群痛点发生率”或“相对整体人群的超额占比”，而不是简单看人数最多的群体。

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st


st.set_page_config(
    page_title="Sephora 面部防晒洞察",
    page_icon="☀️",
    layout="wide",
)

PAIN_CN = {
    "greasy_heavy": "油腻厚重",
    "white_color_cast": "泛白/肤色不适配",
    "pilling": "搓泥",
    "smell": "气味问题",
    "absorb_makeup": "吸收/妆容兼容",
    "protection_failure": "防晒失效",
    "price_value": "价格价值",
    "packaging": "包装问题",
    "breakout_clogging": "爆痘/堵塞",
    "irritation_eyes": "刺激皮肤/辣眼",
}

COLORS = {
    "red": "#D95C5C",
    "orange": "#ED9B40",
    "blue": "#4F7CAC",
    "green": "#4B9C78",
    "light": "#EAF0F6",
}


@st.cache_data
def load_workbook(source):
    return {
        name: pd.read_excel(source, sheet_name=name)
        for name in [
            "痛点总览",
            "产品分析",
            "品牌分析",
            "肤质人群",
            "肤色人群",
            "高风险产品",
            "标杆产品",
        ]
    }


def pct(series):
    return pd.to_numeric(series, errors="coerce") * 100


def format_table(df):
    result = df.copy()
    for col in ["推荐率", "负面率", "主要肤质占比", "主要肤色占比"]:
        if col in result.columns:
            result[col] = pct(result[col]).round(2)
    return result


def segment_pain_rates(df, segment_col):
    pain_cols = [c for c in df.columns if c.startswith("pain_")]
    long = df.melt(
        id_vars=[segment_col, "负面评论数"],
        value_vars=pain_cols,
        var_name="痛点代码",
        value_name="命中评论数",
    )
    long["痛点"] = long["痛点代码"].str.replace("pain_", "", regex=False).map(PAIN_CN)
    long["痛点占该人群负面评论比例"] = np.where(
        long["负面评论数"] > 0,
        long["命中评论数"] / long["负面评论数"] * 100,
        np.nan,
    )
    return long


st.title("☀️ Sephora 面部防晒用户痛点与产品机会分析")
st.caption("从负面评论中识别痛点，并下钻到产品、品牌、肤质和肤色人群。")

uploaded = st.sidebar.file_uploader(
    "上传分析结果 Excel",
    type=["xlsx"],
    help="请选择 sephora_face_sunscreen_analysis.xlsx",
)

default_file = Path("sephora_face_sunscreen_analysis.xlsx")
if uploaded is not None:
    source = uploaded
elif default_file.exists():
    source = default_file
else:
    st.info("请在左侧上传 sephora_face_sunscreen_analysis.xlsx。")
    st.stop()

data = load_workbook(source)
pain = data["痛点总览"].rename(columns={data["痛点总览"].columns[0]: "痛点代码"})
products = data["产品分析"].copy()
brands = data["品牌分析"].copy()
skin_type = data["肤质人群"].copy()
skin_tone = data["肤色人群"].copy()
high_risk = data["高风险产品"].copy()
benchmark = data["标杆产品"].copy()

pain["痛点"] = pain["痛点代码"].map(PAIN_CN).fillna(pain["痛点代码"])
products["负面率_pct"] = pct(products["负面率"])
products["推荐率_pct"] = pct(products["推荐率"])
brands["负面率_pct"] = pct(brands["负面率"])
brands["推荐率_pct"] = pct(brands["推荐率"])

total_reviews = int(products["样本评论数"].sum())
negative_reviews = int(products["负面评论数"].sum())
overall_negative_rate = negative_reviews / total_reviews * 100

tab1, tab2, tab3, tab4, tab5 = st.tabs(
    ["分析总览", "产品风险", "品牌表现", "人群差异", "高风险与标杆"]
)

with tab1:
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("面部防晒评论", f"{total_reviews:,}")
    c2.metric("负面评论", f"{negative_reviews:,}")
    c3.metric("整体负面率", f"{overall_negative_rate:.2f}%")
    c4.metric("覆盖产品", f"{products['product_id'].nunique():,}")

    pain_plot = pain.sort_values("占全部负面评论比例(%)", ascending=True)
    fig = px.bar(
        pain_plot,
        x="占全部负面评论比例(%)",
        y="痛点",
        orientation="h",
        text="占全部负面评论比例(%)",
        color_discrete_sequence=[COLORS["orange"]],
        custom_data=["负面评论数量"],
    )
    fig.update_traces(
        texttemplate="%{text:.1f}%",
        textposition="outside",
        hovertemplate="%{y}<br>占负面评论：%{x:.2f}%<br>评论数：%{customdata[0]:,}<extra></extra>",
    )
    fig.update_layout(
        title="负面评论的核心痛点",
        xaxis_title="占全部负面评论比例（%）",
        yaxis_title=None,
        height=510,
        margin=dict(l=20, r=40, t=60, b=20),
    )
    st.plotly_chart(fig, use_container_width=True)
    st.info(
        "核心结论：油腻厚重是第一大痛点；气味、泛白、搓泥构成第二梯队。"
        "由于一条评论可命中多个痛点，各项比例之和会超过 100%。"
    )

with tab2:
    st.subheader("产品风险定位")
    brands_selected = st.multiselect(
        "筛选品牌",
        options=sorted(products["brand_name"].dropna().unique()),
        default=[],
        placeholder="不选择表示查看全部品牌",
    )
    min_reviews = st.slider(
        "最低样本评论数",
        min_value=0,
        max_value=int(products["样本评论数"].max()),
        value=100,
        step=50,
    )
    filtered = products[products["样本评论数"] >= min_reviews].copy()
    if brands_selected:
        filtered = filtered[filtered["brand_name"].isin(brands_selected)]

    fig = px.scatter(
        filtered,
        x="样本评论数",
        y="负面率_pct",
        size="收藏关注数",
        color="样本平均评分",
        color_continuous_scale="RdYlGn",
        hover_name="product_name",
        hover_data={
            "brand_name": True,
            "商品价格USD": ":.2f",
            "主要痛点": True,
            "样本评论数": ":,",
            "负面率_pct": ":.2f",
            "收藏关注数": ":,",
            "样本平均评分": ":.2f",
        },
        labels={"负面率_pct": "负面率（%）", "样本评论数": "样本评论数"},
        size_max=42,
    )
    fig.add_hline(
        y=overall_negative_rate,
        line_dash="dash",
        annotation_text=f"品类平均 {overall_negative_rate:.2f}%",
    )
    fig.update_layout(height=560, coloraxis_colorbar_title="平均评分")
    st.plotly_chart(fig, use_container_width=True)

    if filtered.empty:
        st.warning("当前筛选条件下没有产品。")
    else:
        selected_product = st.selectbox(
            "查看单个产品",
            filtered.sort_values("负面率_pct", ascending=False)["product_name"],
        )
        row = filtered.loc[filtered["product_name"] == selected_product].iloc[0]
        p1, p2, p3, p4, p5 = st.columns(5)
        p1.metric("品牌", row["brand_name"])
        p2.metric("价格", f"${row['商品价格USD']:.2f}")
        p3.metric("负面率", f"{row['负面率_pct']:.2f}%")
        p4.metric("平均评分", f"{row['样本平均评分']:.2f}")
        p5.metric("主要痛点", str(row["主要痛点"]))

with tab3:
    st.subheader("品牌规模与口碑风险")
    min_brand_reviews = st.slider(
        "品牌最低样本评论数",
        0,
        int(brands["样本评论数"].max()),
        100,
        100,
        key="brand_reviews",
    )
    brand_view = brands[brands["样本评论数"] >= min_brand_reviews].copy()
    fig = px.scatter(
        brand_view,
        x="样本评论数",
        y="负面率_pct",
        size="商品数量",
        color="平均评分",
        text="brand_name",
        color_continuous_scale="RdYlGn",
        hover_name="brand_name",
        hover_data={
            "商品数量": True,
            "商品价格中位数USD": ":.2f",
            "推荐率_pct": ":.2f",
            "样本评论数": ":,",
            "负面率_pct": ":.2f",
            "平均评分": ":.2f",
        },
        labels={"负面率_pct": "负面率（%）", "样本评论数": "样本评论数"},
        size_max=45,
    )
    fig.update_traces(textposition="top center")
    fig.add_hline(y=overall_negative_rate, line_dash="dash")
    fig.update_layout(height=620, coloraxis_colorbar_title="平均评分")
    st.plotly_chart(fig, use_container_width=True)

with tab4:
    st.subheader("不同人群的痛点差异")
    segment_kind = st.radio("分析维度", ["肤质", "肤色"], horizontal=True)
    if segment_kind == "肤质":
        segment_df, segment_col = skin_type, "skin_type"
    else:
        segment_df, segment_col = skin_tone, "skin_tone"

    min_negative = st.slider(
        "人群最低负面评论数",
        0,
        int(segment_df["负面评论数"].max()),
        20,
        10,
        key=f"{segment_kind}_min_negative",
    )
    segment_df = segment_df[segment_df["负面评论数"] >= min_negative].copy()
    long = segment_pain_rates(segment_df, segment_col)
    heat = long.pivot(
        index=segment_col,
        columns="痛点",
        values="痛点占该人群负面评论比例",
    )
    fig = go.Figure(
        data=go.Heatmap(
            z=heat.values,
            x=heat.columns,
            y=heat.index,
            colorscale="YlOrRd",
            colorbar=dict(title="占比 %"),
            hovertemplate=f"{segment_kind}：%{{y}}<br>痛点：%{{x}}<br>占负面评论：%{{z:.2f}}%<extra></extra>",
        )
    )
    fig.update_layout(
        height=max(430, 42 * len(heat.index)),
        xaxis_title="痛点",
        yaxis_title=segment_kind,
        margin=dict(l=20, r=30, t=30, b=100),
    )
    st.plotly_chart(fig, use_container_width=True)
    st.caption("这里的分母是该肤质/肤色人群的负面评论数，用来观察某类人群更容易抱怨什么。")

with tab5:
    left, right = st.columns(2)
    with left:
        st.subheader("高风险产品")
        st.dataframe(
            format_table(high_risk)[
                ["product_name", "brand_name", "样本评论数", "样本平均评分", "负面率", "主要痛点"]
            ],
            use_container_width=True,
            hide_index=True,
        )
    with right:
        st.subheader("标杆产品")
        st.dataframe(
            format_table(benchmark)[
                ["product_name", "brand_name", "样本评论数", "样本平均评分", "负面率", "主要痛点"]
            ],
            use_container_width=True,
            hide_index=True,
        )

    st.success(
        "产品机会：面向油性/混合性肌肤及中深肤色用户，主打轻薄不油腻、隐形不泛白、"
        "妆前不搓泥、低气味和可靠包装；建议规格约 50 mL，价格带约 30–40 美元。"
    )



2026-10-16 10:45:13.786 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-16 10:45:13.802 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-10-16 10:45:13.805 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-16 10:45:18.209 
  command:

    streamlit run C:\Program Files\anaconda\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-10-16 10:45:18.214 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-16 10:45:18.215 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-16 10:45:18.250 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runni